# 07. 열여섯을 고친 것은 전 종목을 고친 것이 아니다

> 2026-09-05 · 이동원 · 결론 문서:
> [데이터파트 v3.6 변경사항](../../docs/데이터파트/version3.6/변경사항.md)

09-04 에 감자 보정을 **정본 코드로 옮겼습니다**([노트북 06](06.스크래치로-고친-것은-고쳐진-것이-아니다.ipynb)).
스크래치 스크립트를 지우고 `ingest/store/adj_price.py` 에 규칙을 심었고, 시험 9건을 붙였고,
극단 수익률 32행이 0행이 됐습니다. 거기서 끝난 줄 알았습니다.

**그런데 실제로 돌린 것은 16종뿐이었습니다.** 스크래치가 찾아 둔 그 16종에만 `--codes` 로
정본 코드를 먹였습니다. 오늘 3,677종 전체를 돌렸더니 **다섯 종목이 더** 나왔고, 그중 하나가
**오검출**이었습니다 — 과거 803행을 1.567배로 부풀리는.

이 노트북은 그 자리를 찾고, 왜 관문 둘이 못 막았는지 재연하고, 무엇으로 갈랐는지
**재고 나서** 임계를 정한 기록입니다.

In [1]:
import json
import sqlite3
import subprocess
import sys
from fractions import Fraction
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

from common.corporate_actions import factor_series  # noqa: E402, I001
from ingest.clients import fdr_data  # noqa: E402, I001
from ingest.store.adj_price import (  # noqa: E402, I001
    CA_BASIS_TOLERANCE,
    CA_PRICE_TOLERANCE,
    CA_SCALE_TOLERANCE,
)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
DB = ROOT / "data" / "krx_cache.db"
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
con.row_factory = sqlite3.Row

print("관문 ① 원가격이 계수만큼 튀었나 :", CA_PRICE_TOLERANCE)
print("관문 ② FDR 이 안 폈나          :", CA_SCALE_TOLERANCE)
print("관문 ③ FDR 이 기준가만큼 폈나   :", CA_BASIS_TOLERANCE, " ← 오늘 새로 넣었습니다")

관문 ① 원가격이 계수만큼 튀었나 : 0.3
관문 ② FDR 이 안 폈나          : 0.1
관문 ③ FDR 이 기준가만큼 폈나   : 0.1  ← 오늘 새로 넣었습니다


## 1. 09-04 가 실제로 돌린 범위

스크래치 보정이 남긴 원장(`reports/fix_capital_reduction_*.json` · 22MB · gitignore)에
**어느 자리를 고쳤는지**가 있습니다. 그 목록이 곧 09-04 에 정본 코드를 먹인 범위입니다.

In [2]:
원장 = sorted((ROOT / "reports").glob("fix_capital_reduction_*.json"))[-1]
기록 = json.loads(원장.read_text(encoding="utf-8"))
구종목 = sorted({자리["code"] for 자리 in 기록["자리"]})
print(f'09-04 스크래치가 찾은 자리 {len(기록["자리"])}개 · 종목 {len(구종목)}종')
print(구종목)

09-04 스크래치가 찾은 자리 17개 · 종목 16종
['001465', '003060', '004555', '007460', '009310', '009415', '012170', '017170', '021045', '050090', '056810', '078860', '079970', '110790', '139050', '176440']


오늘 전 종목을 돌린 뒤의 상태와 나란히 놓습니다.

In [3]:
지금 = [r[0] for r in con.execute(
    "SELECT DISTINCT code FROM daily_price WHERE adj_source LIKE '%ca_fix%' ORDER BY code")]
행 = con.execute(
    "SELECT COUNT(*) FROM daily_price WHERE adj_source LIKE '%ca_fix%'").fetchone()[0]
print(f"지금 보정된 종목 {len(지금)}종 · {행:,}행")
print("09-04 에 없던 종목        :", sorted(set(지금) - set(구종목)))
print("09-04 에 있었는데 지금 없음 :", sorted(set(구종목) - set(지금)))

지금 보정된 종목 20종 · 46,163행
09-04 에 없던 종목        : ['101970', '33637L', '425040', '472850']
09-04 에 있었는데 지금 없음 : []


**네 종목이 늘었습니다.** 전 종목으로 넓히지 않았으면 못 봤을 자리입니다.

그런데 오늘 처음 돌렸을 때는 **다섯**이 늘었습니다. 하나(`003940` 삼양제넥스)를 오늘 새로
넣은 관문 ③이 걷어냈습니다. 아래가 그 이야기입니다.

## 2. 늘어난 자리를 바깥 값으로 검산한다

행 수가 늘었다는 것만으로는 옳은지 알 수 없습니다. **KRX 등락률**로 잽니다 — 우리가 만들지
않은 값이라 정답지 노릇을 합니다.

> KRX 등락률은 `전일대비 / 기준가` 이고 `기준가 = 종가 - 전일대비` 입니다. 소수 2자리로
> 반올림된 `change_rate` 대신 **정수 둘로 만든 정확한 유리수**를 씁니다.

In [4]:
def 자리검산(code, dd):
    """그 날 저장된 수정등락과 KRX 참값을 나란히 돌려준다."""
    rows = [dict(r) for r in con.execute(
        "SELECT bas_dd, name, close, change, adj_close, adj_source "
        "FROM daily_price WHERE code=? ORDER BY bas_dd", (code,))]
    i = next(k for k, r in enumerate(rows) if r["bas_dd"] == dd)
    앞, 이 = rows[i - 1], rows[i]
    기준가 = 이["close"] - 이["change"]
    return {
        "종목": code, "이름": 이["name"], "날짜": dd,
        "저장수정등락": 이["adj_close"] / 앞["adj_close"] - 1,
        "KRX참값": float(Fraction(int(이["change"]), int(기준가))),
        "출처": 이["adj_source"],
    }


새자리 = [("33637L", "20240108"), ("101970", "20250328"),
          ("472850", "20260102"), ("425040", "20240102")]
표 = pd.DataFrame([자리검산(c, d) for c, d in 새자리])
표["차이"] = (표["저장수정등락"] - 표["KRX참값"]).abs()
표

,종목,이름,날짜,저장수정등락,KRX참값,출처,차이
0,33637L,솔루스첨단소재2우B,20240108,0.142322,0.142322,fdr,8.326673e-17
1,101970,우양에이치씨,20250328,0.105150,0.105150,fdr,1.387779e-17
2,472850,폰드그룹,20260102,-0.063604,-0.063604,fdr,1.387779e-17
3,425040,티이엠씨,20240102,-0.027132,-0.027132,fdr,4.857226e-17


**네 자리 모두 차이가 0 입니다.** 새로 잡은 보정이 KRX 와 소수점까지 맞습니다.

다섯 번째가 `003940` 삼양제넥스였습니다. 지금 DB 에는 보정이 없으므로(관문 ③이 막았으므로)
그 자리는 아래에서 따로 재연합니다.

## 3. 다섯 번째 — 삼양제넥스 2013-03-25

원문부터 봅니다. 앞 사흘이 거래정지(`volume 0`)이고, 재개일에 상장주식수가 줄었습니다.

In [5]:
원문 = pd.DataFrame([dict(r) for r in con.execute(
    "SELECT bas_dd, close, change, volume, listed_shares, adj_close, adj_source "
    "FROM daily_price WHERE code='003940' AND bas_dd BETWEEN '20130320' AND '20130328' "
    "ORDER BY bas_dd")])
원문["기준가"] = 원문["close"] - 원문["change"]
원문["KRX"] = 원문["change"] / 원문["기준가"]
원문[["bas_dd", "close", "change", "기준가", "KRX", "volume",
     "listed_shares", "adj_close", "adj_source"]]

,bas_dd,close,change,기준가,KRX,volume,listed_shares,adj_close,adj_source
0,20130320,68900,0,68900,0.000000,0,2985917,70500.0,fdr
1,20130321,68900,0,68900,0.000000,0,2985917,70500.0,fdr
2,20130322,68900,0,68900,0.000000,0,2985917,70500.0,fdr
3,20130325,78000,7500,70500,0.106383,79073,1905907,78000.0,fdr
4,20130326,78000,0,78000,0.000000,17546,1905907,78000.0,fdr
5,20130327,78200,200,78000,0.002564,4740,1905907,78200.0,fdr
6,20130328,80600,2400,78200,0.030691,10887,1905907,80600.0,fdr


세 숫자를 나란히 놓으면 답이 보입니다.

| | 값 | |
|---|---:|---|
| 상장주식수 배율 | 1,905,907 / 2,985,917 = **0.6383** | 계수는 그 역수 **1.5667** |
| KRX 기준가비 | 70,500 / 68,900 = **1.0232** | 가격은 거의 안 끊겼다 |

감자였다면 기준가가 `68,900 × 1.5667 ≈ 108,000` 이어야 합니다. **인적분할입니다** —
회사가 쪼개져 주식수가 줄었을 뿐, 남은 주식의 가치는 그대로입니다.

In [6]:
주식수배율 = Fraction(1905907, 2985917)
계수 = 1 / 주식수배율
기준가비 = Fraction(70500, 68900)
print(f"계수(주식수에서)   {float(계수):.4f}")
print(f"KRX 기준가비       {float(기준가비):.4f}")
print(f"감자였다면 기준가   {68900 * float(계수):,.0f}원 — 실제 기준가는 70,500원")

계수(주식수에서)   1.5667
KRX 기준가비       1.0232
감자였다면 기준가   107,943원 — 실제 기준가는 70,500원


## 4. 왜 관문 둘이 못 막았나

관문 ①②를 그 자리에 그대로 대 봅니다.

In [7]:
앞종가, 종가 = 68900, 78000
원가격비 = Fraction(종가, 앞종가)
관문1 = abs(float(원가격비 / 계수) - 1)

# FDR 이 그 날 실제로 준 값 — 지금 DB 의 adj_close 가 곧 FDR 값이다(보정이 없으므로).
FDR앞 = con.execute(
    "SELECT adj_close FROM daily_price WHERE code='003940' AND bas_dd='20130322'").fetchone()[0]
FDR뒤 = con.execute(
    "SELECT adj_close FROM daily_price WHERE code='003940' AND bas_dd='20130325'").fetchone()[0]
실제 = (Fraction(str(FDR앞)) / 앞종가) / (Fraction(str(FDR뒤)) / 종가)
관문2 = abs(float(실제) - 1)

판정1 = "통과" if 관문1 <= CA_PRICE_TOLERANCE else "차단"
판정2 = "통과" if 관문2 <= CA_SCALE_TOLERANCE else "차단"
print(f"관문 ①  |{float(원가격비):.4f} / {float(계수):.4f} - 1| = {관문1:.4f}"
      f"   <= {CA_PRICE_TOLERANCE}  → {판정1}")
print(f"관문 ②  |{float(실제):.4f} - 1|           = {관문2:.4f}"
      f"   <= {CA_SCALE_TOLERANCE}  → {판정2}")
print()
print(f"보정했다면 그날 수정등락 {float(원가격비 / 계수) - 1:+.4f}"
      f" · KRX 참값 {7500 / 70500:+.4f}")

관문 ①  |1.1321 / 1.5667 - 1| = 0.2774   <= 0.3  → 통과
관문 ②  |1.0232 - 1|           = 0.0232   <= 0.1  → 통과

보정했다면 그날 수정등락 -0.2774 · KRX 참값 +0.1064


관문 ①은 **0.2774 로 임계 0.30 안쪽**입니다. 0.026 차이로 들어왔습니다. 관문 ②는 FDR 이
편 폭이 2.3% 라 "아무것도 안 했다" 로 읽혔습니다.

🔴 관문 ①이 견주는 대상이 잘못돼 있습니다. `원가격비/계수 - 1` 은 **보정한 뒤 그 날의
수정등락 그 자체**인데, 관문은 그것을 **0** 과 견줍니다. 정답은 0 이 아니라 KRX 등락률입니다 —
감자일에도 주가는 움직입니다.

## 5. 결정적 증거 — FDR 은 정지일에 KRX 기준가를 준다

FDR 이 편 폭 `1.0232` 와 KRX 기준가비 `1.0232` 가 **소수 다섯 자리까지 같습니다.**
우연이 아닙니다.

In [8]:
print(f"FDR 이 편 폭   {float(실제):.6f}")
print(f"KRX 기준가비   {float(기준가비):.6f}")
print(f"비             {float(실제 / 기준가비):.6f}")
print()
print("정지일에 FDR 이 준 값 —")
for r in con.execute(
        "SELECT bas_dd, close, adj_close FROM daily_price WHERE code='003940' "
        "AND bas_dd BETWEEN '20130320' AND '20130322' ORDER BY bas_dd"):
    print(f"  {r[0]}  원문 종가 {r[1]:,} → FDR {r[2]:,.0f}")

FDR 이 편 폭   1.023222
KRX 기준가비   1.023222
비             1.000000

정지일에 FDR 이 준 값 —
  20130320  원문 종가 68,900 → FDR 70,500
  20130321  원문 종가 68,900 → FDR 70,500
  20130322  원문 종가 68,900 → FDR 70,500


**FDR 은 정지일에 원문 종가 68,900 이 아니라 KRX 기준가 70,500 을 줍니다.** 이 사건을 알고
이미 폈다는 뜻입니다. 인적분할이니 그게 맞는 조정이고, **틀린 것은 FDR 도 관문도 아니라
우리 계수**였습니다.

## 6. 임계는 재고 나서 적는다 — 세 지표를 22자리에 대 본다

숫자를 먼저 고르고 맞는지 보는 것과 다릅니다. 관문이 실제로 발동한 자리 전부에서 세 후보
지표를 재고, **틈이 벌어진 것**을 고릅니다.

In [9]:
발동자리 = [
    ("012170", "20250306"), ("012170", "20260828"), ("009310", "20260511"),
    ("003060", "20260508"), ("007460", "20260508"), ("021045", "20240925"),
    ("009415", "20240712"), ("001465", "20240417"), ("079970", "20240821"),
    ("078860", "20231227"), ("050090", "20230508"), ("139050", "20230628"),
    ("110790", "20231020"), ("33637L", "20240108"), ("056810", "20130327"),
    ("176440", "20191121"), ("101970", "20250328"), ("004555", "20120514"),
    ("472850", "20260102"), ("017170", "20110404"), ("425040", "20240102"),
    ("003940", "20130325"),   # ← 오늘 걷어낸 자리
]


# 🔴 ③ 은 **FDR 이 실제로 편 폭**(`실제`)이 있어야 잰다. 지금 DB 의 `adj_close` 는
#    보정이 들어간 값이라 되돌릴 수 없으므로(보정하면 `실제` 가 계수와 같아진다),
#    FDR 을 다시 받아 원래 배율을 쓴다. 종목당 1콜이다.
def 지표(code, dd):
    rows = [dict(r) for r in con.execute(
        "SELECT bas_dd, name, open, high, low, close, change, volume, listed_shares "
        "FROM daily_price WHERE code=? ORDER BY bas_dd", (code,))]
    i = next(k for k, r in enumerate(rows) if r["bas_dd"] == dd)
    f = factor_series(rows)[i]
    앞종가, 종가 = rows[i - 1]["close"], rows[i]["close"]
    보정후 = float(Fraction(int(종가), int(앞종가)) / f) - 1
    기준가 = 종가 - rows[i]["change"]
    krx = float(Fraction(int(rows[i]["change"]), int(기준가)))
    기준가비 = Fraction(int(기준가), int(앞종가))

    fdr = FDR캐시.setdefault(code, fdr_data.fetch_adjusted(code))
    앞F = (fdr.get(rows[i - 1]["bas_dd"]) or {}).get("adj_close")
    이F = (fdr.get(dd) or {}).get("adj_close")
    실제 = ((Fraction(str(앞F)) / 앞종가) / (Fraction(str(이F)) / 종가)
            if 앞F and 이F else None)
    return {
        "종목": code, "이름": rows[i]["name"], "날짜": dd,
        "계수": float(f), "KRX": krx,
        "FDR이_편_폭": float(실제) if 실제 is not None else float("nan"),
        "①": abs(보정후),
        "②": abs(보정후 - krx),
        "③": abs(float(실제 / 기준가비) - 1) if 실제 is not None else float("nan"),
    }


FDR캐시 = {}
재기 = pd.DataFrame([지표(c, d) for c, d in 발동자리])
재기.sort_values("③")[["종목", "이름", "날짜", "계수", "KRX",
                        "FDR이_편_폭", "①", "②", "③"]].head(8)

,종목,이름,날짜,계수,KRX,FDR이_편_폭,①,②,③
21,003940,삼양제넥스,20130325,1.566665,0.106383,1.023222,0.277398,3.837807e-01,0.000000
6,009415,태영건설우,20240712,2.003376,0.000000,1.000000,0.001685,1.684916e-03,0.500000
14,056810,위다스,20130327,3.000000,0.000000,1.000000,0.001927,1.926735e-03,0.667308
10,050090,비케이홀딩스,20230508,4.000000,0.299801,1.000000,0.239899,5.990250e-02,0.737922
18,472850,폰드그룹,20260102,0.562624,-0.063604,1.000000,0.063604,1.387779e-17,0.777385
17,004555,대우송도개발1우,20120514,6.974182,0.000000,1.000000,0.283070,2.830700e-01,0.800000
1,012170,아센디오,20260828,5.000001,-0.009358,1.000000,0.009358,1.956804e-07,0.800000
2,009310,참엔지니어링,20260511,5.000001,0.000000,1.000000,0.000939,9.387842e-04,0.800188


> 📌 ③이 코드가 실제로 쓰는 `|실제 / 기준가비 - 1|` 입니다. **0 에 가까울수록 "FDR 이 이미
> KRX 기준가만큼 폈다"** 는 뜻이라 **건드리면 안 되는** 자리입니다. `FDR이_편_폭` 칸을 함께
> 보면 정상 자리는 전부 **1.0 언저리**(= FDR 이 아무것도 안 했다)인데 삼양제넥스만
> **1.0232**(= KRX 기준가만큼 폈다)인 것이 보입니다.

지표별로 **삼양제넥스와 가장 가까운 정상 자리 사이의 여유**를 봅니다.

In [10]:
삼양 = 재기[재기["종목"] == "003940"].iloc[0]
정상 = 재기[재기["종목"] != "003940"]

# ①② 는 "클수록 의심" 이라 정상 쪽 최댓값과, ③ 은 "작을수록 이미 폈다" 라 최솟값과 견준다.
for 이름, 방향, 쪽 in (("①", "클수록 의심", "max"),
                       ("②", "클수록 의심", "max"),
                       ("③", "작을수록 이미 폈다", "min")):
    내값 = 삼양[이름]
    누구 = 정상.loc[정상[이름].idxmax() if 쪽 == "max" else 정상[이름].idxmin()]
    if 쪽 == "max":
        판정 = ("🔴 못 가른다 — 정상 쪽이 더 의심스럽다"
                if 내값 < 누구[이름] else f"여유 {내값 / 누구[이름]:.2f}배")
    else:
        판정 = f"틈 {누구[이름] - 내값:.4f} (임계 {CA_BASIS_TOLERANCE} 가 그 사이)"
    print(f"{이름} ({방향:<16}) 삼양제넥스 {내값:.4f} · "
          f"가장 가까운 정상 {누구[이름]:.4f} ({누구['이름']})")
    print(f"{'':<22} → {판정}")

① (클수록 의심          ) 삼양제넥스 0.2774 · 가장 가까운 정상 0.2831 (대우송도개발1우)
                       → 🔴 못 가른다 — 정상 쪽이 더 의심스럽다
② (클수록 의심          ) 삼양제넥스 0.3838 · 가장 가까운 정상 0.2831 (대우송도개발1우)
                       → 여유 1.36배
③ (작을수록 이미 폈다      ) 삼양제넥스 0.0000 · 가장 가까운 정상 0.5000 (태영건설우)
                       → 틈 0.5000 (임계 0.1 가 그 사이)


**①로는 아예 못 가릅니다** — 삼양제넥스(0.2774)가 정상 자리 `004555`
대우송도개발1우(0.2831)보다 **오히려 덜 의심스럽게** 나옵니다. 임계를 조이면 진짜 감자를
먼저 잃습니다. ②도 여유가 1.36배뿐이라 마찬가지입니다(004555 는 진짜 감자인데 거래정지
중이라 기준가가 무의미한 자리입니다).

**③은 0.0000 과 0.5000 사이가 통째로 비어 있습니다.** 임계를 낮은 쪽에 붙여 `0.10` 으로
뒀습니다 — 정상 자리를 잃는 것이 오검출보다 비싸기 때문입니다. 조정을 놓치면 수익률이
틀리지만, **없는 조정을 만들면 멀쩡한 과거 전체를 망칩니다.**

## 7. 관문 ③을 넣고 다시 깔았다

`ingest/store/adj_price.py::_fix_unadjusted_actions` 에 관문 하나를 더 뒀습니다.

```python
# ③ FDR 이 이미 KRX 기준가만큼 폈나 — 그러면 틀린 것은 계수다.
기준가비 = Fraction(int(종가 - 전일대비), int(앞종가))
if 기준가비 != 1 and abs(float(실제 / 기준가비) - 1) <= CA_BASIS_TOLERANCE:
    continue
```

`기준가비 != 1` 조건이 필요합니다 — KRX 가 기준가를 안 바꾼 날에는 대조할 것이 없고,
**감자인데 등락률이 `0.00%` 인 자리가 실재**하기 때문입니다(참엔지니어링 2026-05-11).
그 조건이 없으면 진짜 감자를 놓칩니다. 시험으로 박아 뒀습니다.

In [11]:
결과 = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_adj_price.py", "-q", "--no-header"],
    cwd=ROOT, capture_output=True, text=True)
print(결과.stdout.strip().splitlines()[-1])

24 passed in 2.05s


## 8. 결과 — 바깥 값으로 확인한다

In [12]:
삼양검산 = 자리검산("003940", "20130325")
차이 = abs(삼양검산["저장수정등락"] - 삼양검산["KRX참값"])
print(f'삼양제넥스 그날 수정등락 {삼양검산["저장수정등락"]:+.4%}')
print(f'KRX 참값                 {삼양검산["KRX참값"]:+.4%}')
print(f"차이                     {차이:.6%}")
print(f'출처                     {삼양검산["출처"]}  ← ca_fix 가 없다')
print()
for 라벨, sql in (
        ("ca_fix 행", "SELECT COUNT(*) FROM daily_price WHERE adj_source LIKE '%ca_fix%'"),
        ("ca_fix 종목",
         "SELECT COUNT(DISTINCT code) FROM daily_price WHERE adj_source LIKE '%ca_fix%'"),
        ("adj_close 결측", "SELECT COUNT(*) FROM daily_price WHERE adj_close IS NULL"),
        ("원가격 행", "SELECT COUNT(*) FROM daily_price"),
        ("원가격 close 합", "SELECT SUM(close) FROM daily_price"),
):
    print(f"{라벨:<16} {con.execute(sql).fetchone()[0]:,}")

삼양제넥스 그날 수정등락 +10.6383%
KRX 참값                 +10.6383%
차이                     0.000000%
출처                     fdr  ← ca_fix 가 없다



ca_fix 행         46,163


ca_fix 종목        20


adj_close 결측     0
원가격 행            9,223,644


원가격 close 합      235,166,505,589


| | 09-04 (16종만) | 오늘 첫 적재 (전 종목) | 관문 ③ 뒤 |
|---|---:|---:|---:|
| ca_fix 행 | 43,778 | 46,966 | **46,163** |
| ca_fix 종목 | 16 | 21 | **20** |
| 삼양제넥스 그날 오차 | — | 0.3838 | **0.000000** |

원가격 지문(행 9,223,644 · `close` 합 235,166,505,589)은 세 번 다 같습니다 —
원본은 안 건드렸습니다.

## 9. 곁가지 — `verify_base_info` §9 임계는 2배로 둔다

§9 는 "주식수가 **2배 이상** 변한 자리"만 봅니다. 1.5배로 내릴지 검토하려 했는데, 오늘 그
답이 실측으로 나왔습니다.

In [13]:
구간 = 재기[재기["계수"].between(1.5, 2.0) | 재기["계수"].between(0.5, 1 / 1.5)]
print("계수가 1.5~2.0 (또는 그 역수) 구간인 발동 자리")
print(구간[["종목", "이름", "날짜", "계수", "KRX", "FDR이_편_폭", "③"]].to_string(index=False))

계수가 1.5~2.0 (또는 그 역수) 구간인 발동 자리
    종목         이름       날짜       계수       KRX  FDR이_편_폭        ③
021045     대호특수강우 20240925 0.500000 -0.001319  1.000000 1.000000
110790    크리스에프앤씨 20231020 0.500000 -0.003165  1.000000 1.000000
33637L 솔루스첨단소재2우B 20240108 0.500000  0.142322  1.000000 1.000000
472850       폰드그룹 20260102 0.562624 -0.063604  1.000000 0.777385
425040       티이엠씨 20240102 0.500000 -0.027132  1.000000 1.000000
003940      삼양제넥스 20130325 1.566665  0.106383  1.023222 0.000000


그 구간에 실재하는 것은 **삼양제넥스 1.5667 하나이고, 그것이 감자가 아닙니다.**
임계를 내리면 오경보만 하나 늘어납니다. **2배로 둡니다.**

## 10. 남는 한계

- 🔴 **자본변동 당일 하루는 여전히 부정확합니다.** 계수를 상장주식수 배율에서 얻는데 KRX
  기준가는 다른 값입니다(아센디오 2025-03-06: 우리 −10.22% vs KRX +3.25%).
  **다음날부터는 정확합니다.** 거래정지 중 종가가 직전 실질가가 아니라 붙들고 있던 값이라
  생기는 일입니다.
- **FDR 이 조정을 하루 일찍 놓는 자리**가 둘 있습니다(티이엠씨 2023-12-28 ·
  폰드그룹 2025-12-30). 우리 보정이 만든 것이 아니라 FDR 원값의 성질입니다.
- 저가주는 **1원 눈금**이 수익률로 읽힙니다 — 아이오케이의 2023년 43쌍이 그것입니다.

## 오늘 배운 것

> **열여섯을 고친 것은 전 종목을 고친 것이 아닙니다.** 정본 코드를 만들어도 그것을 스크래치가
> 찾아 둔 범위에만 먹이면, 규칙이 아니라 목록을 옮긴 것입니다.

> **임계는 재고 나서 적습니다.** 관문 ①의 0.30 은 09-04 에 41자리를 재서 정한 값이었는데,
> 전 종목으로 넓히니 0.2774 가 나타났습니다. 여유 0.026 은 임계가 아니라 우연입니다.
> 여유가 **1.02배**인 지표와 **틈이 빈** 지표는 다릅니다.